In [0]:
 from pyspark.sql import functions as F

# Caminho do arquivo RAW
path_listings = "/Volumes/workspace/default/airbnb_sp/raw/listings.csv.gz"

# Leitura do arquivo CSV
df_listings_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .load(path_listings)
)

# Exibir as primeiras linhas
display(df_listings_raw.limit(10))

# Informações básicas
print("Quantidade de registros:", df_listings_raw.count())
print("Quantidade de colunas:", len(df_listings_raw.columns))

print("\nColunas:")
print(df_listings_raw.columns)

In [0]:
from pyspark.sql import functions as F

# Criação da camada Bronze
df_bronze_listings = (
    df_listings_raw
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn(
        "_source_file",
        F.lit("listings.csv.gz")
    )
)

# Gravação como tabela Delta
(
    df_bronze_listings.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.bronze_listings")
)

print("Tabela Bronze criada com sucesso!")

In [0]:
 # Validação da tabela Bronze

df_bronze_check = spark.table("workspace.default.bronze_listings")

print("Quantidade de registros:", df_bronze_check.count())
print("Quantidade de colunas:", len(df_bronze_check.columns))

display(
    df_bronze_check.select(
        "id",
        "name",
        "neighbourhood_cleansed",
        "room_type",
        "price",
        "accommodates",
        "bedrooms",
        "beds",
        "availability_365",
        "number_of_reviews",
        "review_scores_rating",
        "_ingested_at",
        "_source_file"
    ).limit(10)
)

In [0]:
 # ============================================================
# AUDITORIA INICIAL DE QUALIDADE — BRONZE
# ============================================================

from pyspark.sql import functions as F

df = spark.table("workspace.default.bronze_listings")

# 1. Nulos nos principais atributos
colunas_principais = [
    "id",
    "name",
    "neighbourhood_cleansed",
    "room_type",
    "property_type",
    "price",
    "accommodates",
    "bedrooms",
    "beds",
    "availability_365",
    "number_of_reviews",
    "review_scores_rating"
]

resultado_nulos = []

for coluna in colunas_principais:
    nulos = df.filter(
        F.col(coluna).isNull() | (F.trim(F.col(coluna)) == "")
    ).count()

    resultado_nulos.append(
        (coluna, nulos, round((nulos / df.count()) * 100, 2))
    )

df_nulos = spark.createDataFrame(
    resultado_nulos,
    ["atributo", "qtd_nulos", "percentual_nulos"]
)

display(df_nulos.orderBy(F.desc("percentual_nulos")))

In [0]:
 # ============================================================
# AUDITORIA DE DUPLICIDADES — BRONZE
# ============================================================

df = spark.table("workspace.default.bronze_listings")

total_registros = df.count()

# Verifica IDs duplicados
ids_duplicados = (
    df.groupBy("id")
      .count()
      .filter(F.col("count") > 1)
)

qtd_ids_duplicados = ids_duplicados.count()

# Quantidade de registros que fazem parte de duplicidades
registros_duplicados = (
    ids_duplicados
    .select(F.sum("count"))
    .collect()[0][0]
)

print("Total de registros:", total_registros)
print("IDs duplicados:", qtd_ids_duplicados)
print("Registros envolvidos em duplicidades:", registros_duplicados or 0)

display(ids_duplicados.orderBy(F.desc("count")).limit(20))

In [0]:
 # ============================================================
# AUDITORIA DE TIPOS E VALORES
# ============================================================

df = spark.table("workspace.default.bronze_listings")

# Mostrar os tipos atuais das principais colunas
print("TIPOS DOS ATRIBUTOS PRINCIPAIS:\n")

for coluna in [
    "id",
    "price",
    "accommodates",
    "bedrooms",
    "beds",
    "availability_365",
    "number_of_reviews",
    "review_scores_rating"
]:
    print(f"{coluna}: {df.schema[coluna].dataType}")

# ============================================================
# CONVERSÃO TEMPORÁRIA PARA AUDITORIA
# ============================================================

df_audit = (
    df
    .withColumn(
        "price_num",
        F.regexp_replace(
            F.regexp_replace(F.col("price"), r"[$,]", ""),
            r"\.",
            "."
        ).cast("double")
    )
    .withColumn("accommodates_num", F.col("accommodates").cast("double"))
    .withColumn("bedrooms_num", F.col("bedrooms").cast("double"))
    .withColumn("beds_num", F.col("beds").cast("double"))
    .withColumn(
        "availability_num",
        F.col("availability_365").cast("double")
    )
    .withColumn(
        "reviews_num",
        F.col("number_of_reviews").cast("double")
    )
    .withColumn(
        "rating_num",
        F.col("review_scores_rating").cast("double")
    )
)

# ============================================================
# VALORES INVÁLIDOS
# ============================================================

print("\nVALORES POTENCIALMENTE INVÁLIDOS:\n")

checks = [
    ("price <= 0", F.col("price_num") <= 0),
    ("accommodates <= 0", F.col("accommodates_num") <= 0),
    ("bedrooms < 0", F.col("bedrooms_num") < 0),
    ("beds < 0", F.col("beds_num") < 0),
    ("availability fora de 0-365",
     (F.col("availability_num") < 0) |
     (F.col("availability_num") > 365)),
    ("number_of_reviews < 0", F.col("reviews_num") < 0),
    ("rating fora de 0-5",
     (F.col("rating_num") < 0) |
     (F.col("rating_num") > 5))
]

for nome, condicao in checks:
    quantidade = df_audit.filter(condicao).count()
    print(f"{nome}: {quantidade}")

In [0]:
 # ============================================================
# AUDITORIA DE PREÇO — DISTRIBUIÇÃO E OUTLIERS
# ============================================================

df = spark.table("workspace.default.bronze_listings")

df_preco = (
    df
    .withColumn(
        "price_num",
        F.regexp_replace(F.col("price"), r"[$,]", "").cast("double")
    )
    .filter(F.col("price_num").isNotNull())
)

# Estatísticas principais
estatisticas = df_preco.select(
    F.count("price_num").alias("registros_com_preco"),
    F.min("price_num").alias("preco_minimo"),
    F.expr("percentile_approx(price_num, 0.25)").alias("q1"),
    F.expr("percentile_approx(price_num, 0.50)").alias("mediana"),
    F.expr("percentile_approx(price_num, 0.75)").alias("q3"),
    F.max("price_num").alias("preco_maximo"),
    F.avg("price_num").alias("preco_medio")
)

display(estatisticas)

# ============================================================
# IDENTIFICAÇÃO DE OUTLIERS PELO MÉTODO DO IQR
# ============================================================

q = estatisticas.collect()[0]

q1 = q["q1"]
q3 = q["q3"]

iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Limite inferior:", limite_inferior)
print("Limite superior:", limite_superior)

outliers = df_preco.filter(
    (F.col("price_num") < limite_inferior) |
    (F.col("price_num") > limite_superior)
)

print("Quantidade de outliers:", outliers.count())

display(
    outliers
    .select(
        "id",
        "name",
        "neighbourhood_cleansed",
        "room_type",
        "price",
        "price_num"
    )
    .orderBy(F.desc("price_num"))
    .limit(20)
)

In [0]:
# ============================================================
# CAMADA SILVER — LIMPEZA E PADRONIZAÇÃO
# ============================================================

from pyspark.sql import functions as F

df_bronze = spark.table("workspace.default.bronze_listings")

# ------------------------------------------------------------
# 1. Conversão dos principais atributos para tipos adequados
# ------------------------------------------------------------

df_silver = (
    df_bronze

    # Identificação
    .withColumn("id", F.col("id").cast("long"))

    # Preço: remove símbolo $ e separador de milhar
    .withColumn(
        "price",
        F.regexp_replace(F.col("price"), r"[$,]", "").cast("double")
    )

    # Variáveis numéricas
    .withColumn("accommodates", F.col("accommodates").cast("int"))
    .withColumn("bedrooms", F.col("bedrooms").cast("double"))
    .withColumn("beds", F.col("beds").cast("double"))
    .withColumn("availability_365", F.col("availability_365").cast("int"))
    .withColumn("number_of_reviews", F.col("number_of_reviews").cast("int"))
    .withColumn(
        "review_scores_rating",
        F.col("review_scores_rating").cast("double")
    )

    # --------------------------------------------------------
    # 2. Padronização de textos
    # --------------------------------------------------------

    .withColumn(
        "neighbourhood_cleansed",
        F.trim(F.col("neighbourhood_cleansed"))
    )
    .withColumn(
        "room_type",
        F.trim(F.col("room_type"))
    )
    .withColumn(
        "property_type",
        F.trim(F.col("property_type"))
    )

    # --------------------------------------------------------
    # 3. Criação da flag de outlier de preço
    # --------------------------------------------------------

    .withColumn(
        "price_outlier",
        F.when(
            F.col("price") > 698.75,
            F.lit(True)
        ).otherwise(F.lit(False))
    )

    # --------------------------------------------------------
    # 4. Indicador de anúncio sem avaliação
    # --------------------------------------------------------

    .withColumn(
        "has_rating",
        F.when(
            F.col("review_scores_rating").isNotNull(),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
)

# ------------------------------------------------------------
# 5. Remover registros sem ID
# ------------------------------------------------------------

df_silver = df_silver.filter(
    F.col("id").isNotNull()
)

print("Registros Silver:", df_silver.count())
print("Colunas Silver:", len(df_silver.columns))

In [0]:
 # ============================================================
# VALIDAÇÃO DA CAMADA SILVER
# ============================================================

print("TIPOS APÓS A TRANSFORMAÇÃO:\n")

for coluna in [
    "id",
    "price",
    "accommodates",
    "bedrooms",
    "beds",
    "availability_365",
    "number_of_reviews",
    "review_scores_rating"
]:
    print(f"{coluna}: {df_silver.schema[coluna].dataType}")

# ------------------------------------------------------------
# VALIDAÇÃO DOS OUTLIERS
# ------------------------------------------------------------

print("\nOUTLIERS DE PREÇO:")

display(
    df_silver
    .groupBy("price_outlier")
    .count()
    .orderBy("price_outlier")
)

# ------------------------------------------------------------
# VALIDAÇÃO DOS ANÚNCIOS COM/SEM AVALIAÇÃO
# ------------------------------------------------------------

print("\nSTATUS DE AVALIAÇÃO:")

display(
    df_silver
    .groupBy("has_rating")
    .count()
    .orderBy("has_rating")
)

# ------------------------------------------------------------
# AMOSTRA FINAL DA SILVER
# ------------------------------------------------------------

display(
    df_silver.select(
        "id",
        "neighbourhood_cleansed",
        "room_type",
        "property_type",
        "price",
        "accommodates",
        "bedrooms",
        "beds",
        "availability_365",
        "number_of_reviews",
        "review_scores_rating",
        "price_outlier",
        "has_rating"
    ).limit(10)
)

In [0]:
 # ============================================================
# GRAVAÇÃO DA CAMADA SILVER
# ============================================================

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.silver_listings")
)

print("Tabela Silver criada com sucesso!")

In [0]:
 # ============================================================
# ETAPA 3 — PREPARAÇÃO DA CAMADA GOLD
# ============================================================

from pyspark.sql import functions as F

df_gold_base = (
    df_silver
    .select(
        "id",
        "listing_url",
        "name",
        "neighbourhood_cleansed",
        "latitude",
        "longitude",
        "property_type",
        "room_type",
        "accommodates",
        "bathrooms",
        "bedrooms",
        "beds",
        "price",
        "minimum_nights",
        "maximum_nights",
        "availability_30",
        "availability_60",
        "availability_90",
        "availability_365",
        "number_of_reviews",
        "review_scores_rating",
        "host_is_superhost",
        "instant_bookable",
        "last_scraped"
    )
    .dropDuplicates(["id"])
)

print("Registros disponíveis para Gold:", df_gold_base.count())
print("Colunas disponíveis para Gold:", len(df_gold_base.columns))

In [0]:
 # ============================================================
# ETAPA 3.1 — DIMENSÕES DA CAMADA GOLD
# ============================================================

# DIMENSÃO 1 — ANÚNCIO
dim_listing = (
    df_gold_base
    .select(
        "id",
        "listing_url",
        "name",
        "property_type",
        "accommodates",
        "bathrooms",
        "bedrooms",
        "beds",
        "minimum_nights",
        "maximum_nights",
        "host_is_superhost",
        "instant_bookable"
    )
    .dropDuplicates(["id"])
)

# DIMENSÃO 2 — LOCALIZAÇÃO
dim_neighbourhood = (
    df_gold_base
    .select(
        "neighbourhood_cleansed",
        "latitude",
        "longitude"
    )
    .dropDuplicates(["neighbourhood_cleansed"])
)

# DIMENSÃO 3 — TIPO DE ACOMODAÇÃO
dim_room_type = (
    df_gold_base
    .select("room_type")
    .dropDuplicates()
    .filter(F.col("room_type").isNotNull())
)

print("DIMENSÕES CRIADAS")
print("dim_listing:", dim_listing.count(), "registros")
print("dim_neighbourhood:", dim_neighbourhood.count(), "registros")
print("dim_room_type:", dim_room_type.count(), "registros")

In [0]:
 # ============================================================
# 13. CONFERÊNCIA DAS DIMENSÕES PARA O MODELO GOLD
# ============================================================

print("COLUNAS DA DIM_LISTING:")
print(dim_listing.columns)

print("\nCOLUNAS DA DIM_NEIGHBOURHOOD:")
print(dim_neighbourhood.columns)

print("\nCOLUNAS DA DIM_ROOM_TYPE:")
print(dim_room_type.columns)

print("\nAMOSTRA DIM_LISTING:")
display(dim_listing.limit(5))

print("\nAMOSTRA DIM_NEIGHBOURHOOD:")
display(dim_neighbourhood.limit(5))

print("\nAMOSTRA DIM_ROOM_TYPE:")
display(dim_room_type.limit(5))

In [0]:
 # ============================================================
# 14. CRIAÇÃO DA TABELA FATO — GOLD
# ============================================================

from pyspark.sql import functions as F

fact_listing = (
    df_gold_base
    .select(
        "id",
        "neighbourhood_cleansed",
        "room_type",
        "price",
        "accommodates",
        "bedrooms",
        "beds",
        "availability_30",
        "availability_60",
        "availability_90",
        "availability_365",
        "number_of_reviews",
        "review_scores_rating",
        "host_is_superhost",
        "instant_bookable",
        "last_scraped"
    )
    .dropDuplicates(["id"])
)

print("TABELA FATO CRIADA")
print("Registros:", fact_listing.count())
print("Colunas:", len(fact_listing.columns))

display(
    fact_listing.limit(10)
)

In [0]:
 # ============================================================
# 15. GRAVAÇÃO DA TABELA FATO NO GOLD
# ============================================================

(
    fact_listing
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.fact_listing")
)

print("Tabela Gold criada com sucesso!")
print("Registros:", spark.table("workspace.default.fact_listing").count())

In [0]:
# ============================================================
# 16. IDENTIFICAÇÃO DOS SCHEMAS DISPONÍVEIS
# ============================================================

print("CATÁLOGOS:")
spark.sql("SHOW CATALOGS").show(truncate=False)

print("\nSCHEMAS DO CATÁLOGO WORKSPACE:")
spark.sql("SHOW SCHEMAS IN workspace").show(truncate=False) 

In [0]:
 # ============================================================
# 17. GOLD - PREÇO MÉDIO POR BAIRRO
# ============================================================

from pyspark.sql import functions as F

gold_neighbourhood_price = (
    fact_listing
    .filter(
        (F.col("price") > 0) &
        F.col("neighbourhood_cleansed").isNotNull()
    )
    .groupBy("neighbourhood_cleansed")
    .agg(
        F.count("*").alias("total_listings"),
        F.round(F.avg("price"), 2).alias("preco_medio"),
        F.round(F.expr("percentile_approx(price, 0.5)"), 2).alias("preco_mediano"),
        F.round(F.min("price"), 2).alias("preco_minimo"),
        F.round(F.max("price"), 2).alias("preco_maximo")
    )
    .orderBy(F.desc("preco_medio"))
)

(
    gold_neighbourhood_price
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.gold_neighbourhood_price")
)

print("Tabela Gold de preços por bairro criada!")
print("Bairros:", gold_neighbourhood_price.count())

display(gold_neighbourhood_price.limit(20))

In [0]:
 # ============================================================
# 18. GOLD - PREÇO POR BAIRRO COM REPRESENTATIVIDADE
# ============================================================

gold_neighbourhood_price_filtered = (
    gold_neighbourhood_price
    .filter(F.col("total_listings") >= 20)
    .orderBy(F.desc("preco_medio"))
)

print("BAIRROS COM PELO MENOS 20 ANÚNCIOS:")
print(
    "Quantidade de bairros:",
    gold_neighbourhood_price_filtered.count()
)

print("\n10 BAIRROS COM MAIOR PREÇO MÉDIO:")

display(
    gold_neighbourhood_price_filtered
    .orderBy(F.desc("preco_medio"))
    .limit(10)
)

print("\n10 BAIRROS COM MENOR PREÇO MÉDIO:")

display(
    gold_neighbourhood_price_filtered
    .orderBy(F.asc("preco_medio"))
    .limit(10)
)

In [0]:
 # ============================================================
# 19. ANÁLISE — PREÇO POR TIPO DE ACOMODAÇÃO
# ============================================================

from pyspark.sql import functions as F

gold_room_type_price = (
    fact_listing
    .filter(
        (F.col("price") > 0) &
        F.col("room_type").isNotNull()
    )
    .groupBy("room_type")
    .agg(
        F.count("*").alias("total_anuncios"),
        F.round(F.avg("price"), 2).alias("preco_medio"),
        F.round(F.expr("percentile_approx(price, 0.5)"), 2).alias("preco_mediano"),
        F.round(F.min("price"), 2).alias("preco_minimo"),
        F.round(F.max("price"), 2).alias("preco_maximo"),
        F.round(F.avg("accommodates"), 2).alias("hospedes_medio"),
        F.round(F.avg("availability_365"), 2).alias("disponibilidade_media_365")
    )
    .orderBy(F.desc("preco_medio"))
)

print("PREÇO MÉDIO POR TIPO DE ACOMODAÇÃO:")
print("")

display(gold_room_type_price)

In [0]:
 # ============================================================
# 20. ANÁLISE DE QUALIDADE DOS DADOS - GOLD
# ============================================================

from pyspark.sql import functions as F

print("============================================================")
print("ANÁLISE DE QUALIDADE DOS DADOS")
print("============================================================")

# ------------------------------------------------------------
# 1. VOLUME TOTAL
# ------------------------------------------------------------

total_registros = fact_listing.count()

print(f"\nTotal de registros: {total_registros:,}")
print(f"Total de atributos: {len(fact_listing.columns)}")

# ------------------------------------------------------------
# 2. DUPLICIDADES DE ID
# ------------------------------------------------------------

duplicados = (
    fact_listing
    .groupBy("id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"\nIDs duplicados: {duplicados:,}")

# ------------------------------------------------------------
# 3. VALORES NULOS
# ------------------------------------------------------------

colunas_qualidade = [
    "id",
    "neighbourhood_cleansed",
    "room_type",
    "price",
    "accommodates",
    "bedrooms",
    "beds",
    "availability_365",
    "number_of_reviews",
    "review_scores_rating"
]

# Mantém somente colunas que realmente existem
colunas_existentes = [
    c for c in colunas_qualidade
    if c in fact_listing.columns
]

resultado_nulos = []

for coluna in colunas_existentes:

    nulos = fact_listing.filter(
        F.col(coluna).isNull()
    ).count()

    percentual = (nulos / total_registros) * 100

    resultado_nulos.append(
        (coluna, nulos, percentual)
    )

df_nulos = spark.createDataFrame(
    resultado_nulos,
    ["atributo", "valores_nulos", "percentual_nulo"]
)

print("\nNULOS POR ATRIBUTO:")
display(
    df_nulos.orderBy(
        F.desc("percentual_nulo")
    )
)

# ------------------------------------------------------------
# 4. VALORES INVÁLIDOS
# ------------------------------------------------------------

print("\nVALORES INVÁLIDOS:")

if "price" in fact_listing.columns:
    preco_invalido = fact_listing.filter(
        (F.col("price") <= 0) |
        F.col("price").isNull()
    ).count()

    print(f"Preço <= 0 ou nulo: {preco_invalido:,}")

if "accommodates" in fact_listing.columns:
    hospedes_invalidos = fact_listing.filter(
        (F.col("accommodates") <= 0) |
        F.col("accommodates").isNull()
    ).count()

    print(f"Capacidade <= 0 ou nula: {hospedes_invalidos:,}")

if "availability_365" in fact_listing.columns:
    disponibilidade_invalida = fact_listing.filter(
        (F.col("availability_365") < 0) |
        (F.col("availability_365") > 365)
    ).count()

    print(
        f"Disponibilidade fora de 0-365: "
        f"{disponibilidade_invalida:,}"
    )

if "number_of_reviews" in fact_listing.columns:
    reviews_invalidas = fact_listing.filter(
        F.col("number_of_reviews") < 0
    ).count()

    print(
        f"Número de avaliações negativo: "
        f"{reviews_invalidas:,}"
    )

if "review_scores_rating" in fact_listing.columns:
    rating_invalido = fact_listing.filter(
        (F.col("review_scores_rating") < 0) |
        (F.col("review_scores_rating") > 5)
    ).count()

    print(
        f"Rating fora de 0-5: "
        f"{rating_invalido:,}"
    )

# ------------------------------------------------------------
# 5. DISTRIBUIÇÃO DOS TIPOS DE ACOMODAÇÃO
# ------------------------------------------------------------

print("\nDISTRIBUIÇÃO DOS TIPOS DE ACOMODAÇÃO:")

display(
    fact_listing
    .groupBy("room_type")
    .agg(
        F.count("*").alias("total_anuncios")
    )
    .withColumn(
        "percentual",
        F.round(
            F.col("total_anuncios") /
            F.lit(total_registros) * 100,
            2
        )
    )
    .orderBy(F.desc("total_anuncios"))
)

print("\n============================================================")
print("QUALIDADE DOS DADOS CONCLUÍDA")
print("============================================================")

In [0]:
 # ============================================================
# 21. INVESTIGAÇÃO DE PREÇOS NULOS E OUTLIERS
# ============================================================

from pyspark.sql import functions as F

print("============================================================")
print("INVESTIGAÇÃO DA VARIÁVEL PREÇO")
print("============================================================")

# ------------------------------------------------------------
# 1. SEPARAR PREÇOS NULOS, ZERO E POSITIVOS
# ------------------------------------------------------------

print("\nCLASSIFICAÇÃO DOS PREÇOS:")

precos_nulos = fact_listing.filter(
    F.col("price").isNull()
).count()

precos_zero = fact_listing.filter(
    F.col("price") == 0
).count()

precos_negativos = fact_listing.filter(
    F.col("price") < 0
).count()

precos_validos = fact_listing.filter(
    F.col("price") > 0
).count()

print(f"Preços nulos:       {precos_nulos:,}")
print(f"Preços iguais a 0:  {precos_zero:,}")
print(f"Preços negativos:   {precos_negativos:,}")
print(f"Preços válidos:     {precos_validos:,}")

# ------------------------------------------------------------
# 2. ESTATÍSTICAS DOS PREÇOS VÁLIDOS
# ------------------------------------------------------------

print("\nESTATÍSTICAS DOS PREÇOS VÁLIDOS:")

estatisticas_preco = (
    fact_listing
    .filter(F.col("price") > 0)
    .select(
        F.round(F.min("price"), 2).alias("preco_minimo"),
        F.round(F.expr("percentile(price, 0.25)"), 2).alias("Q1"),
        F.round(F.expr("percentile(price, 0.50)"), 2).alias("mediana"),
        F.round(F.expr("percentile(price, 0.75)"), 2).alias("Q3"),
        F.round(F.max("price"), 2).alias("preco_maximo"),
        F.round(F.avg("price"), 2).alias("media")
    )
)

display(estatisticas_preco)

# ------------------------------------------------------------
# 3. IDENTIFICAR OS 20 MAIORES PREÇOS
# ------------------------------------------------------------

print("\n20 MAIORES PREÇOS:")

display(
    fact_listing
    .filter(F.col("price") > 0)
    .select(
        "id",
        "neighbourhood_cleansed",
        "room_type",
        "price",
        "accommodates",
        "bedrooms",
        "beds"
    )
    .orderBy(F.desc("price"))
    .limit(20)
)

# ------------------------------------------------------------
# 4. IDENTIFICAR OUTLIERS PELO MÉTODO DO IQR
# ------------------------------------------------------------

quantis = (
    fact_listing
    .filter(F.col("price") > 0)
    .select(
        F.expr("percentile(price, 0.25)").alias("Q1"),
        F.expr("percentile(price, 0.75)").alias("Q3")
    )
    .collect()[0]
)

q1 = quantis["Q1"]
q3 = quantis["Q3"]

iqr = q3 - q1
limite_superior = q3 + (1.5 * iqr)

print("\nANÁLISE DE OUTLIERS - MÉTODO IQR")
print(f"Q1: {q1:.2f}")
print(f"Q3: {q3:.2f}")
print(f"IQR: {iqr:.2f}")
print(f"Limite superior: {limite_superior:.2f}")

outliers = (
    fact_listing
    .filter(F.col("price") > limite_superior)
)

print(
    f"\nQuantidade de outliers de preço: "
    f"{outliers.count():,}"
)

print("\nEXEMPLOS DE OUTLIERS:")

display(
    outliers
    .select(
        "id",
        "neighbourhood_cleansed",
        "room_type",
        "price",
        "accommodates",
        "bedrooms",
        "beds"
    )
    .orderBy(F.desc("price"))
    .limit(20)
)

print("\n============================================================")
print("INVESTIGAÇÃO CONCLUÍDA")
print("============================================================")

In [0]:
 # ============================================================
# 22. PERGUNTA 3 - CARACTERÍSTICAS DO IMÓVEL X PREÇO
# ============================================================

from pyspark.sql import functions as F

print("============================================================")
print("PERGUNTA 3")
print("CARACTERÍSTICAS DO IMÓVEL X PREÇO")
print("============================================================")

# ------------------------------------------------------------
# BASE ANALÍTICA
# Remove apenas preços nulos/zero.
# Os outliers continuam preservados.
# ------------------------------------------------------------

df_preco = (
    fact_listing
    .filter(F.col("price") > 0)
)

# ------------------------------------------------------------
# 1. PREÇO POR CAPACIDADE
# ------------------------------------------------------------

print("\n1. PREÇO MÉDIO E MEDIANO POR CAPACIDADE:")

preco_accommodates = (
    df_preco
    .filter(
        (F.col("accommodates") >= 1) &
        (F.col("accommodates") <= 16)
    )
    .groupBy("accommodates")
    .agg(
        F.count("*").alias("total_anuncios"),
        F.round(F.avg("price"), 2).alias("preco_medio"),
        F.round(
            F.expr("percentile(price, 0.50)"),
            2
        ).alias("preco_mediano")
    )
    .orderBy("accommodates")
)

display(preco_accommodates)

# ------------------------------------------------------------
# 2. PREÇO POR NÚMERO DE QUARTOS
# ------------------------------------------------------------

print("\n2. PREÇO MÉDIO E MEDIANO POR NÚMERO DE QUARTOS:")

preco_bedrooms = (
    df_preco
    .filter(
        (F.col("bedrooms") >= 0) &
        (F.col("bedrooms") <= 10)
    )
    .groupBy("bedrooms")
    .agg(
        F.count("*").alias("total_anuncios"),
        F.round(F.avg("price"), 2).alias("preco_medio"),
        F.round(
            F.expr("percentile(price, 0.50)"),
            2
        ).alias("preco_mediano")
    )
    .orderBy("bedrooms")
)

display(preco_bedrooms)

# ------------------------------------------------------------
# 3. PREÇO POR NÚMERO DE CAMAS
# ------------------------------------------------------------

print("\n3. PREÇO MÉDIO E MEDIANO POR NÚMERO DE CAMAS:")

preco_beds = (
    df_preco
    .filter(
        (F.col("beds") >= 0) &
        (F.col("beds") <= 12)
    )
    .groupBy("beds")
    .agg(
        F.count("*").alias("total_anuncios"),
        F.round(F.avg("price"), 2).alias("preco_medio"),
        F.round(
            F.expr("percentile(price, 0.50)"),
            2
        ).alias("preco_mediano")
    )
    .orderBy("beds")
)

display(preco_beds)

# ------------------------------------------------------------
# 4. CORRELAÇÕES
# ------------------------------------------------------------

print("\n4. CORRELAÇÃO ENTRE CARACTERÍSTICAS E PREÇO:")

if "accommodates" in df_preco.columns:
    corr_accommodates = df_preco.stat.corr(
        "accommodates",
        "price"
    )
    print(
        f"Correlação capacidade x preço: "
        f"{corr_accommodates:.4f}"
    )

if "bedrooms" in df_preco.columns:
    corr_bedrooms = df_preco.stat.corr(
        "bedrooms",
        "price"
    )
    print(
        f"Correlação quartos x preço: "
        f"{corr_bedrooms:.4f}"
    )

if "beds" in df_preco.columns:
    corr_beds = df_preco.stat.corr(
        "beds",
        "price"
    )
    print(
        f"Correlação camas x preço: "
        f"{corr_beds:.4f}"
    )

print("\n============================================================")
print("PERGUNTA 3 CONCLUÍDA")
print("============================================================")

In [0]:
 # ============================================================
# 20. PERGUNTA 4 — AVALIAÇÃO x PREÇO
# ============================================================

from pyspark.sql import functions as F

print("=" * 60)
print("PERGUNTA 4 — AVALIAÇÃO x PREÇO")
print("=" * 60)

# Seleciona apenas registros válidos para a análise
rating_price = (
    fact_listing
    .filter(
        (F.col("price") > 0) &
        (F.col("rating").isNotNull()) &
        (F.col("rating") > 0) &
        (F.col("rating") <= 5)
    )
)

print("Registros utilizados:", rating_price.count())

# Criação de faixas de avaliação
rating_bands = (
    rating_price
    .withColumn(
        "faixa_avaliacao",
        F.when(F.col("rating") < 3.0, "Abaixo de 3,0")
         .when(F.col("rating") < 4.0, "3,0 a 3,9")
         .when(F.col("rating") < 4.5, "4,0 a 4,4")
         .when(F.col("rating") < 4.8, "4,5 a 4,7")
         .otherwise("4,8 a 5,0")
    )
)

# Preço médio e mediano por faixa
gold_rating_price = (
    rating_bands
    .groupBy("faixa_avaliacao")
    .agg(
        F.count("*").alias("total_anuncios"),
        F.round(F.avg("price"), 2).alias("preco_medio"),
        F.round(F.expr("percentile_approx(price, 0.5)"), 2).alias("preco_mediano"),
        F.round(F.avg("rating"), 2).alias("avaliacao_media")
    )
    .orderBy("faixa_avaliacao")
)

print("\nPREÇO POR FAIXA DE AVALIAÇÃO:")
display(gold_rating_price)

# Correlação entre avaliação e preço
correlacao_rating_preco = (
    rating_price
    .select(
        F.corr("rating", "price").alias("correlacao")
    )
    .collect()[0]["correlacao"]
)

print("\nCORRELAÇÃO ENTRE AVALIAÇÃO E PREÇO:")
print(f"Correlação rating x preço: {correlacao_rating_preco:.4f}")

print("\n" + "=" * 60)
print("PERGUNTA 4 CONCLUÍDA")
print("=" * 60)

In [0]:
 # ============================================================
# 20A. IDENTIFICAR COLUNAS DE AVALIAÇÃO
# ============================================================

print("=" * 60)
print("COLUNAS DA TABELA fact_listing")
print("=" * 60)

for coluna in fact_listing.columns:
    print(coluna)

print("\n" + "=" * 60)
print("COLUNAS RELACIONADAS A AVALIAÇÕES")
print("=" * 60)

colunas_rating = [
    coluna for coluna in fact_listing.columns
    if any(palavra in coluna.lower()
           for palavra in ["rating", "review", "score", "avali"])
]

print(colunas_rating)

In [0]:
 # ============================================================
# 21. PERGUNTA 4 — AVALIAÇÃO x PREÇO
# ============================================================

from pyspark.sql import functions as F

print("=" * 60)
print("PERGUNTA 4 — AVALIAÇÃO x PREÇO")
print("=" * 60)

# Selecionar anúncios com preço e avaliação válidos
rating_price = (
    fact_listing
    .filter(
        (F.col("price") > 0) &
        (F.col("review_scores_rating").isNotNull()) &
        (F.col("review_scores_rating") > 0) &
        (F.col("review_scores_rating") <= 5)
    )
)

print("Registros utilizados:", rating_price.count())

# Criar faixas de avaliação
rating_bands = (
    rating_price
    .withColumn(
        "faixa_avaliacao",
        F.when(F.col("review_scores_rating") < 3.0, "Abaixo de 3,0")
         .when(F.col("review_scores_rating") < 4.0, "3,0 a 3,9")
         .when(F.col("review_scores_rating") < 4.5, "4,0 a 4,4")
         .when(F.col("review_scores_rating") < 4.8, "4,5 a 4,7")
         .otherwise("4,8 a 5,0")
    )
)

# Preço médio e mediano por faixa de avaliação
gold_rating_price = (
    rating_bands
    .groupBy("faixa_avaliacao")
    .agg(
        F.count("*").alias("total_anuncios"),
        F.round(F.avg("price"), 2).alias("preco_medio"),
        F.round(
            F.expr("percentile_approx(price, 0.5)"), 2
        ).alias("preco_mediano"),
        F.round(
            F.avg("review_scores_rating"), 2
        ).alias("avaliacao_media")
    )
    .orderBy("faixa_avaliacao")
)

print("\nPREÇO POR FAIXA DE AVALIAÇÃO:")
display(gold_rating_price)

# Correlação entre avaliação e preço
correlacao_rating_preco = (
    rating_price
    .select(
        F.corr(
            "review_scores_rating",
            "price"
        ).alias("correlacao")
    )
    .collect()[0]["correlacao"]
)

print("\nCORRELAÇÃO ENTRE AVALIAÇÃO E PREÇO:")

if correlacao_rating_preco is not None:
    print(
        f"Correlação avaliação x preço: "
        f"{correlacao_rating_preco:.4f}"
    )
else:
    print("Não foi possível calcular a correlação.")

print("\n" + "=" * 60)
print("PERGUNTA 4 CONCLUÍDA")
print("=" * 60)

In [0]:
 # ============================================================
# PERGUNTA 5 — DISPONIBILIDADE POR BAIRRO E TIPO DE ACOMODAÇÃO
# ============================================================

from pyspark.sql import functions as F

print("=" * 70)
print("PERGUNTA 5 — DISPONIBILIDADE DOS ANÚNCIOS")
print("=" * 70)


# ------------------------------------------------------------
# 1. DISPONIBILIDADE MÉDIA POR TIPO DE ACOMODAÇÃO
# ------------------------------------------------------------

print("\nDISPONIBILIDADE MÉDIA POR TIPO DE ACOMODAÇÃO:")
print("-" * 70)

gold_availability_room = (
    fact_listing
    .groupBy("room_type")
    .agg(
        F.count("*").alias("total_anuncios"),
        F.round(F.avg("availability_30"), 2).alias("disponibilidade_media_30"),
        F.round(F.avg("availability_60"), 2).alias("disponibilidade_media_60"),
        F.round(F.avg("availability_90"), 2).alias("disponibilidade_media_90"),
        F.round(F.avg("availability_365"), 2).alias("disponibilidade_media_365")
    )
    .orderBy(F.desc("disponibilidade_media_365"))
)

display(gold_availability_room)


# ------------------------------------------------------------
# 2. DISPONIBILIDADE MÉDIA POR BAIRRO
# ------------------------------------------------------------

print("\nDISPONIBILIDADE MÉDIA POR BAIRRO:")
print("-" * 70)

gold_availability_neighbourhood = (
    fact_listing
    .groupBy("neighbourhood_cleansed")
    .agg(
        F.count("*").alias("total_anuncios"),
        F.round(F.avg("availability_30"), 2).alias("disponibilidade_media_30"),
        F.round(F.avg("availability_60"), 2).alias("disponibilidade_media_60"),
        F.round(F.avg("availability_90"), 2).alias("disponibilidade_media_90"),
        F.round(F.avg("availability_365"), 2).alias("disponibilidade_media_365")
    )
    .filter(F.col("total_anuncios") >= 20)
    .orderBy(F.desc("disponibilidade_media_365"))
)

display(gold_availability_neighbourhood)


# ------------------------------------------------------------
# 3. BAIRROS COM MAIOR DISPONIBILIDADE
# ------------------------------------------------------------

print("\n10 BAIRROS COM MAIOR DISPONIBILIDADE ANUAL:")
print("-" * 70)

top_availability = (
    gold_availability_neighbourhood
    .orderBy(F.desc("disponibilidade_media_365"))
    .limit(10)
)

display(top_availability)


# ------------------------------------------------------------
# 4. BAIRROS COM MENOR DISPONIBILIDADE
# ------------------------------------------------------------

print("\n10 BAIRROS COM MENOR DISPONIBILIDADE ANUAL:")
print("-" * 70)

bottom_availability = (
    gold_availability_neighbourhood
    .orderBy(F.asc("disponibilidade_media_365"))
    .limit(10)
)

display(bottom_availability)


# ------------------------------------------------------------
# 5. CRUZAMENTO: BAIRRO × TIPO DE ACOMODAÇÃO
# ------------------------------------------------------------

print("\nDISPONIBILIDADE POR BAIRRO E TIPO DE ACOMODAÇÃO:")
print("-" * 70)

gold_availability_cross = (
    fact_listing
    .groupBy(
        "neighbourhood_cleansed",
        "room_type"
    )
    .agg(
        F.count("*").alias("total_anuncios"),
        F.round(F.avg("availability_30"), 2).alias("media_30_dias"),
        F.round(F.avg("availability_60"), 2).alias("media_60_dias"),
        F.round(F.avg("availability_90"), 2).alias("media_90_dias"),
        F.round(F.avg("availability_365"), 2).alias("media_365_dias")
    )
    .filter(F.col("total_anuncios") >= 20)
    .orderBy(
        F.desc("media_365_dias")
    )
)

display(gold_availability_cross)


# ------------------------------------------------------------
# 6. SALVAR TABELAS GOLD
# ------------------------------------------------------------

gold_availability_room.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.gold_availability_room")

gold_availability_neighbourhood.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.gold_availability_neighbourhood")

gold_availability_cross.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.gold_availability_cross")


print("\n" + "=" * 70)
print("PERGUNTA 5 CONCLUÍDA")
print("=" * 70)
print("Tabelas Gold de disponibilidade criadas com sucesso.")

In [0]:
 # ============================================================
# PERGUNTA 6 — PERFIS COM MELHOR COMBINAÇÃO
# PREÇO + AVALIAÇÃO + DISPONIBILIDADE
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("=" * 70)
print("PERGUNTA 6 — ANÁLISE DE OPORTUNIDADE")
print("=" * 70)


# ------------------------------------------------------------
# 1. SELECIONAR OS DADOS NECESSÁRIOS
# ------------------------------------------------------------

df_oportunidade = (
    fact_listing
    .select(
        "id",
        "neighbourhood_cleansed",
        "room_type",
        "price",
        "accommodates",
        "bedrooms",
        "beds",
        "availability_365",
        "number_of_reviews",
        "review_scores_rating"
    )
    .filter(
        (F.col("price") > 0) &
        (F.col("review_scores_rating").isNotNull()) &
        (F.col("review_scores_rating") > 0) &
        (F.col("availability_365").isNotNull())
    )
)

print("Registros utilizados:", df_oportunidade.count())


# ------------------------------------------------------------
# 2. CALCULAR OS PERCENTIS
# ------------------------------------------------------------

# Percentil do preço:
# quanto MENOR o preço, melhor para o indicador de oportunidade.

w_price = Window.orderBy(F.col("price"))

df_oportunidade = df_oportunidade.withColumn(
    "percentil_preco",
    F.percent_rank().over(w_price)
)


# Percentil da avaliação:
# quanto MAIOR a avaliação, melhor.

w_rating = Window.orderBy(F.col("review_scores_rating"))

df_oportunidade = df_oportunidade.withColumn(
    "percentil_avaliacao",
    F.percent_rank().over(w_rating)
)


# Percentil da disponibilidade:
# quanto MAIOR a disponibilidade, maior a capacidade
# potencial de receber reservas.

w_availability = Window.orderBy(F.col("availability_365"))

df_oportunidade = df_oportunidade.withColumn(
    "percentil_disponibilidade",
    F.percent_rank().over(w_availability)
)


# ------------------------------------------------------------
# 3. CRIAR O ÍNDICE DE OPORTUNIDADE
# ------------------------------------------------------------

df_oportunidade = df_oportunidade.withColumn(
    "indice_oportunidade",
    F.round(
        (
            (1 - F.col("percentil_preco")) * 0.40
            +
            F.col("percentil_avaliacao") * 0.35
            +
            F.col("percentil_disponibilidade") * 0.25
        ) * 100,
        2
    )
)


# ------------------------------------------------------------
# 4. TOP 20 ANÚNCIOS
# ------------------------------------------------------------

print("\nTOP 20 ANÚNCIOS POR ÍNDICE DE OPORTUNIDADE:")
print("-" * 70)

top_20_oportunidades = (
    df_oportunidade
    .select(
        "id",
        "neighbourhood_cleansed",
        "room_type",
        "price",
        "review_scores_rating",
        "availability_365",
        "number_of_reviews",
        "indice_oportunidade"
    )
    .orderBy(F.desc("indice_oportunidade"))
    .limit(20)
)

display(top_20_oportunidades)


# ------------------------------------------------------------
# 5. PERFIL MÉDIO DOS MELHORES ANÚNCIOS
# ------------------------------------------------------------

print("\nPERFIL MÉDIO DOS 10% MELHORES ANÚNCIOS:")
print("-" * 70)

limite_top_10 = (
    df_oportunidade
    .approxQuantile(
        "indice_oportunidade",
        [0.90],
        0.01
    )[0]
)

top_10 = (
    df_oportunidade
    .filter(F.col("indice_oportunidade") >= limite_top_10)
)

perfil_top_10 = (
    top_10
    .agg(
        F.count("*").alias("total_anuncios"),
        F.round(F.avg("price"), 2).alias("preco_medio"),
        F.round(F.avg("review_scores_rating"), 2).alias("avaliacao_media"),
        F.round(F.avg("availability_365"), 2).alias("disponibilidade_media_365"),
        F.round(F.avg("number_of_reviews"), 2).alias("avaliacoes_recebidas_media"),
        F.round(F.avg("accommodates"), 2).alias("capacidade_media"),
        F.round(F.avg("bedrooms"), 2).alias("quartos_medios")
    )
)

display(perfil_top_10)


# ------------------------------------------------------------
# 6. PERFIL POR TIPO DE ACOMODAÇÃO
# ------------------------------------------------------------

print("\nÍNDICE MÉDIO POR TIPO DE ACOMODAÇÃO:")
print("-" * 70)

oportunidade_room_type = (
    df_oportunidade
    .groupBy("room_type")
    .agg(
        F.count("*").alias("total_anuncios"),
        F.round(F.avg("price"), 2).alias("preco_medio"),
        F.round(F.avg("review_scores_rating"), 2).alias("avaliacao_media"),
        F.round(F.avg("availability_365"), 2).alias("disponibilidade_media"),
        F.round(F.avg("indice_oportunidade"), 2).alias("indice_medio")
    )
    .orderBy(F.desc("indice_medio"))
)

display(oportunidade_room_type)


# ------------------------------------------------------------
# 7. PERFIL POR BAIRRO
# ------------------------------------------------------------

print("\n20 BAIRROS COM MAIOR ÍNDICE MÉDIO DE OPORTUNIDADE:")
print("-" * 70)

oportunidade_neighbourhood = (
    df_oportunidade
    .groupBy("neighbourhood_cleansed")
    .agg(
        F.count("*").alias("total_anuncios"),
        F.round(F.avg("price"), 2).alias("preco_medio"),
        F.round(F.avg("review_scores_rating"), 2).alias("avaliacao_media"),
        F.round(F.avg("availability_365"), 2).alias("disponibilidade_media"),
        F.round(F.avg("indice_oportunidade"), 2).alias("indice_medio")
    )
    .filter(F.col("total_anuncios") >= 20)
    .orderBy(F.desc("indice_medio"))
    .limit(20)
)

display(oportunidade_neighbourhood)


# ------------------------------------------------------------
# 8. SALVAR TABELAS GOLD
# ------------------------------------------------------------

top_20_oportunidades.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.gold_top_oportunidades")

perfil_top_10.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.gold_perfil_oportunidade")

oportunidade_room_type.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.gold_oportunidade_room_type")

oportunidade_neighbourhood.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.gold_oportunidade_neighbourhood")


print("\n" + "=" * 70)
print("PERGUNTA 6 CONCLUÍDA")
print("=" * 70)
print("Tabelas Gold de oportunidade criadas com sucesso.")

In [0]:
 # ============================================================
# RESUMO EXECUTIVO DO MVP
# ============================================================

from pyspark.sql import functions as F

print("=" * 80)
print("RESUMO EXECUTIVO — AIRBNB SÃO PAULO")
print("=" * 80)

# ------------------------------------------------------------
# 1. DIMENSÃO DA BASE
# ------------------------------------------------------------

total_anuncios = fact_listing.count()

print("\n1. DIMENSÃO DA BASE")
print("-" * 80)
print(f"Total de anúncios analisados: {total_anuncios:,}")


# ------------------------------------------------------------
# 2. PREÇO GERAL
# ------------------------------------------------------------

resumo_preco = (
    fact_listing
    .filter(F.col("price") > 0)
    .agg(
        F.round(F.avg("price"), 2).alias("preco_medio"),
        F.round(F.expr("percentile_approx(price, 0.5)"), 2).alias("preco_mediano"),
        F.round(F.min("price"), 2).alias("preco_minimo"),
        F.round(F.max("price"), 2).alias("preco_maximo")
    )
)

print("\n2. PREÇO DOS ANÚNCIOS")
print("-" * 80)
display(resumo_preco)


# ------------------------------------------------------------
# 3. TIPOS DE ACOMODAÇÃO
# ------------------------------------------------------------

print("\n3. DISTRIBUIÇÃO POR TIPO DE ACOMODAÇÃO")
print("-" * 80)

distribuicao = (
    fact_listing
    .groupBy("room_type")
    .agg(
        F.count("*").alias("total_anuncios")
    )
    .withColumn(
        "percentual",
        F.round(
            F.col("total_anuncios") / total_anuncios * 100,
            2
        )
    )
    .orderBy(F.desc("total_anuncios"))
)

display(distribuicao)


# ------------------------------------------------------------
# 4. AVALIAÇÃO
# ------------------------------------------------------------

print("\n4. AVALIAÇÃO DOS ANÚNCIOS")
print("-" * 80)

resumo_rating = (
    fact_listing
    .filter(F.col("review_scores_rating").isNotNull())
    .agg(
        F.count("*").alias("anuncios_com_avaliacao"),
        F.round(
            F.avg("review_scores_rating"),
            2
        ).alias("avaliacao_media")
    )
)

display(resumo_rating)


# ------------------------------------------------------------
# 5. DISPONIBILIDADE
# ------------------------------------------------------------

print("\n5. DISPONIBILIDADE")
print("-" * 80)

resumo_disponibilidade = (
    fact_listing
    .agg(
        F.round(F.avg("availability_30"), 2)
            .alias("media_disponibilidade_30"),
        F.round(F.avg("availability_60"), 2)
            .alias("media_disponibilidade_60"),
        F.round(F.avg("availability_90"), 2)
            .alias("media_disponibilidade_90"),
        F.round(F.avg("availability_365"), 2)
            .alias("media_disponibilidade_365")
    )
)

display(resumo_disponibilidade)


# ------------------------------------------------------------
# 6. CORRELAÇÃO CAPACIDADE × PREÇO
# ------------------------------------------------------------

correlacao_capacidade = (
    fact_listing
    .filter(
        (F.col("price") > 0) &
        (F.col("accommodates") > 0)
    )
    .select(
        F.corr("accommodates", "price")
            .alias("correlacao_capacidade_preco")
    )
)

print("\n6. RELAÇÃO ENTRE CAPACIDADE E PREÇO")
print("-" * 80)

display(correlacao_capacidade)


# ------------------------------------------------------------
# 7. CORRELAÇÃO AVALIAÇÃO × PREÇO
# ------------------------------------------------------------

correlacao_rating = (
    fact_listing
    .filter(
        (F.col("price") > 0) &
        F.col("review_scores_rating").isNotNull()
    )
    .select(
        F.corr(
            "review_scores_rating",
            "price"
        ).alias("correlacao_avaliacao_preco")
    )
)

print("\n7. RELAÇÃO ENTRE AVALIAÇÃO E PREÇO")
print("-" * 80)

display(correlacao_rating)


# ------------------------------------------------------------
# 8. FINALIZAÇÃO
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("RESUMO EXECUTIVO CONCLUÍDO")
print("=" * 80)
print("As principais métricas do MVP foram consolidadas.")

In [0]:
 # ================================================================
# AUDITORIA FINAL DO MVP
# Qualidade, volume, duplicidades, outliers e camadas do pipeline
# ================================================================

from pyspark.sql import functions as F

print("=" * 70)
print("AUDITORIA FINAL DO MVP - INSIDE AIRBNB SÃO PAULO")
print("=" * 70)


# ================================================================
# 1. VOLUME DA TABELA PRINCIPAL
# ================================================================

total_registros = fact_listing.count()

print("\n1. VOLUME DA TABELA PRINCIPAL")
print("-" * 70)
print(f"Total de registros em fact_listing: {total_registros:,}")
print(f"Total de colunas: {len(fact_listing.columns)}")


# ================================================================
# 2. QUALIDADE DOS PRINCIPAIS ATRIBUTOS
# ================================================================

colunas_qualidade = [
    "id",
    "neighbourhood_cleansed",
    "room_type",
    "price",
    "accommodates",
    "bedrooms",
    "beds",
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_365",
    "number_of_reviews",
    "review_scores_rating"
]

colunas_existentes = [
    c for c in colunas_qualidade
    if c in fact_listing.columns
]

print("\n2. QUALIDADE DOS DADOS - NULOS POR ATRIBUTO")
print("-" * 70)

expr_nulos = [
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in colunas_existentes
]

nulos = fact_listing.agg(*expr_nulos).collect()[0]

for c in colunas_existentes:
    quantidade = nulos[c]
    percentual = (quantidade / total_registros) * 100
    print(f"{c:25} Nulos: {quantidade:8,} | {percentual:6.2f}%")


# ================================================================
# 3. DUPLICIDADES
# ================================================================

print("\n3. DUPLICIDADES")
print("-" * 70)

if "id" in fact_listing.columns:

    total_ids = fact_listing.select("id").count()

    ids_unicos = (
        fact_listing
        .select("id")
        .distinct()
        .count()
    )

    duplicados = total_ids - ids_unicos

    print(f"Registros analisados: {total_ids:,}")
    print(f"IDs únicos:          {ids_unicos:,}")
    print(f"Registros duplicados: {duplicados:,}")

else:
    print("Coluna id não encontrada.")


# ================================================================
# 4. VALORES INVÁLIDOS
# ================================================================

print("\n4. VALORES INVÁLIDOS")
print("-" * 70)

if "price" in fact_listing.columns:
    preco_invalido = fact_listing.filter(
        (F.col("price") <= 0) | F.col("price").isNull()
    ).count()

    print(f"Preço <= 0 ou nulo:              {preco_invalido:,}")

if "accommodates" in fact_listing.columns:
    capacidade_invalida = fact_listing.filter(
        (F.col("accommodates") <= 0) | F.col("accommodates").isNull()
    ).count()

    print(f"Capacidade <= 0 ou nula:         {capacidade_invalida:,}")

if "availability_365" in fact_listing.columns:
    disponibilidade_invalida = fact_listing.filter(
        (F.col("availability_365") < 0) |
        (F.col("availability_365") > 365) |
        F.col("availability_365").isNull()
    ).count()

    print(f"Disponibilidade fora de 0-365:   {disponibilidade_invalida:,}")

if "number_of_reviews" in fact_listing.columns:
    reviews_invalidas = fact_listing.filter(
        (F.col("number_of_reviews") < 0) |
        F.col("number_of_reviews").isNull()
    ).count()

    print(f"Nº de avaliações negativo/nulo:  {reviews_invalidas:,}")

if "review_scores_rating" in fact_listing.columns:
    rating_invalido = fact_listing.filter(
        (F.col("review_scores_rating") < 0) |
        (F.col("review_scores_rating") > 5)
    ).count()

    print(f"Rating fora de 0-5:              {rating_invalido:,}")


# ================================================================
# 5. OUTLIERS DE PREÇO
# Método: intervalo interquartil (IQR)
# ================================================================

print("\n5. OUTLIERS DE PREÇO - MÉTODO IQR")
print("-" * 70)

if "price" in fact_listing.columns:

    quartis = (
        fact_listing
        .filter(F.col("price") > 0)
        .approxQuantile("price", [0.25, 0.75], 0.01)
    )

    q1 = quartis[0]
    q3 = quartis[1]
    iqr = q3 - q1

    limite_inferior = q1 - (1.5 * iqr)
    limite_superior = q3 + (1.5 * iqr)

    outliers = fact_listing.filter(
        (F.col("price") < limite_inferior) |
        (F.col("price") > limite_superior)
    ).count()

    print(f"Q1:                         R$ {q1:,.2f}")
    print(f"Q3:                         R$ {q3:,.2f}")
    print(f"IQR:                        R$ {iqr:,.2f}")
    print(f"Limite superior:            R$ {limite_superior:,.2f}")
    print(f"Quantidade de outliers:     {outliers:,}")
    print(f"Percentual de outliers:     {(outliers/total_registros)*100:.2f}%")


# ================================================================
# 6. DISTRIBUIÇÃO DOS TIPOS DE ACOMODAÇÃO
# ================================================================

print("\n6. DISTRIBUIÇÃO DOS TIPOS DE ACOMODAÇÃO")
print("-" * 70)

if "room_type" in fact_listing.columns:

    distribuicao = (
        fact_listing
        .groupBy("room_type")
        .agg(
            F.count("*").alias("total_anuncios")
        )
        .withColumn(
            "percentual",
            F.round(
                F.col("total_anuncios") / total_registros * 100,
                2
            )
        )
        .orderBy(F.desc("total_anuncios"))
    )

    display(distribuicao)


# ================================================================
# 7. CAMADAS DO PIPELINE
# ================================================================

print("\n7. CAMADAS DO PIPELINE")
print("-" * 70)

tabelas_pipeline = [
    "bronze_listings",
    "silver_listings",
    "fact_listing",
    "dim_neighbourhood",
    "dim_room_type",
    "gold_neighbourhood_price",
    "gold_room_type_price",
    "gold_availability",
    "gold_opportunity"
]

for tabela in tabelas_pipeline:

    try:

        existe = spark.catalog.tableExists(
            f"workspace.default.{tabela}"
        )

        if existe:

            qtd = spark.table(
                f"workspace.default.{tabela}"
            ).count()

            print(f"✓ {tabela:30} {qtd:,} registros")

        else:

            print(f"— {tabela:30} não encontrada")

    except Exception as e:

        print(f"? {tabela:30} verificação não disponível")


# ================================================================
# 8. RESUMO EXECUTIVO
# ================================================================

print("\n" + "=" * 70)
print("RESUMO EXECUTIVO DA AUDITORIA")
print("=" * 70)

print(f"""
Dataset analisado:
Inside Airbnb - São Paulo

Tabela principal:
fact_listing

Registros analisados:
{total_registros:,}

Colunas analisadas:
{len(fact_listing.columns)}

Pipeline:
Bronze → Silver → Gold → Análise

Qualidade:
Foram verificadas ausências, duplicidades,
valores inválidos e outliers de preço.

Análises de negócio:
✓ Preço por bairro
✓ Preço por tipo de acomodação
✓ Características × preço
✓ Avaliação × preço
✓ Disponibilidade
✓ Índice de oportunidade

STATUS DO MVP:
✓ Pipeline executado
✓ Dados transformados
✓ Camadas analíticas criadas
✓ Qualidade avaliada
✓ Perguntas de negócio analisadas
""")

print("=" * 70)
print("AUDITORIA FINAL CONCLUÍDA COM SUCESSO")
print("=" * 70)

In [0]:
 # ================================================================
# INVENTÁRIO REAL DAS TABELAS DO MVP
# Identificação das tabelas efetivamente criadas no Databricks
# ================================================================

print("=" * 75)
print("INVENTÁRIO REAL DO PIPELINE - WORKSPACE.DEFAULT")
print("=" * 75)

tabelas = spark.catalog.listTables("workspace.default")

tabelas_mvp = []

for tabela in tabelas:

    nome = tabela.name

    # Considera objetos relacionados ao nosso MVP
    palavras_chave = [
        "bronze",
        "silver",
        "fact",
        "dim",
        "gold"
    ]

    if any(palavra in nome.lower() for palavra in palavras_chave):

        try:
            df = spark.table(f"workspace.default.{nome}")
            quantidade = df.count()
            colunas = len(df.columns)

            tabelas_mvp.append(
                (nome, quantidade, colunas, tabela.tableType)
            )

        except Exception:
            tabelas_mvp.append(
                (nome, "erro", "erro", tabela.tableType)
            )


print("\nTABELAS ENCONTRADAS:")
print("-" * 75)

if tabelas_mvp:

    for nome, quantidade, colunas, tipo in sorted(tabelas_mvp):

        print(
            f"✓ {nome:35} "
            f"| Registros: {str(quantidade):>10} "
            f"| Colunas: {str(colunas):>3} "
            f"| Tipo: {tipo}"
        )

else:

    print("Nenhuma tabela do pipeline foi encontrada.")


# ================================================================
# RESUMO
# ================================================================

print("\n" + "=" * 75)
print(f"TOTAL DE OBJETOS DO MVP ENCONTRADOS: {len(tabelas_mvp)}")
print("=" * 75)

In [0]:
 # ============================================================
# MODELO DIMENSIONAL — MVP INSIDE AIRBNB SÃO PAULO
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("=" * 70)
print("CRIANDO MODELO DIMENSIONAL")
print("=" * 70)

# ------------------------------------------------------------
# 1. DIMENSÃO BAIRRO
# ------------------------------------------------------------

dim_neighbourhood = (
    fact_listing
    .select("neighbourhood_cleansed")
    .where(F.col("neighbourhood_cleansed").isNotNull())
    .distinct()
    .withColumn(
        "neighbourhood_key",
        F.row_number().over(
            Window.orderBy("neighbourhood_cleansed")
        )
    )
    .select(
        "neighbourhood_key",
        "neighbourhood_cleansed"
    )
)

dim_neighbourhood.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.dim_neighbourhood")


# ------------------------------------------------------------
# 2. DIMENSÃO TIPO DE ACOMODAÇÃO
# ------------------------------------------------------------

dim_room_type = (
    fact_listing
    .select("room_type")
    .where(F.col("room_type").isNotNull())
    .distinct()
    .withColumn(
        "room_type_key",
        F.row_number().over(
            Window.orderBy("room_type")
        )
    )
    .select(
        "room_type_key",
        "room_type"
    )
)

dim_room_type.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.dim_room_type")


# ------------------------------------------------------------
# 3. VERIFICAÇÃO
# ------------------------------------------------------------

print("\nDIM_NEIGHBOURHOOD:")
display(
    spark.table("workspace.default.dim_neighbourhood")
    .orderBy("neighbourhood_key")
)

print("\nDIM_ROOM_TYPE:")
display(
    spark.table("workspace.default.dim_room_type")
    .orderBy("room_type_key")
)

print("\n" + "=" * 70)
print("MODELO DIMENSIONAL CRIADO COM SUCESSO")
print("=" * 70)

In [0]:
  # ============================================================
# FACT LISTING DIMENSIONAL
# ============================================================

from pyspark.sql import functions as F

print("=" * 70)
print("CRIANDO FACT_LISTING_DIMENSIONAL")
print("=" * 70)

# ------------------------------------------------------------
# Recupera as dimensões
# ------------------------------------------------------------

dim_bairro = spark.table(
    "workspace.default.dim_neighbourhood"
)

dim_tipo = spark.table(
    "workspace.default.dim_room_type"
)


# ------------------------------------------------------------
# Junta a tabela fato às dimensões
# ------------------------------------------------------------

fact_listing_dim = (
    fact_listing.alias("f")

    .join(
        dim_bairro.alias("b"),
        F.col("f.neighbourhood_cleansed") ==
        F.col("b.neighbourhood_cleansed"),
        "left"
    )

    .join(
        dim_tipo.alias("t"),
        F.col("f.room_type") ==
        F.col("t.room_type"),
        "left"
    )

    .select(
        F.col("f.id").alias("listing_id"),

        F.col("b.neighbourhood_key"),
        F.col("b.neighbourhood_cleansed"),

        F.col("t.room_type_key"),
        F.col("t.room_type"),

        F.col("f.price"),
        F.col("f.accommodates"),
        F.col("f.bedrooms"),
        F.col("f.beds"),

        F.col("f.availability_30"),
        F.col("f.availability_60"),
        F.col("f.availability_90"),
        F.col("f.availability_365"),

        F.col("f.number_of_reviews"),
        F.col("f.review_scores_rating"),

        F.col("f.host_is_superhost"),
        F.col("f.instant_bookable"),
        F.col("f.last_scraped")
    )
)


# ------------------------------------------------------------
# Grava a tabela Gold dimensional
# ------------------------------------------------------------

fact_listing_dim.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.fact_listing_dimensional"
    )


# ------------------------------------------------------------
# AUDITORIA
# ------------------------------------------------------------

total = fact_listing_dim.count()

nulos_bairro = fact_listing_dim \
    .filter(F.col("neighbourhood_key").isNull()) \
    .count()

nulos_tipo = fact_listing_dim \
    .filter(F.col("room_type_key").isNull()) \
    .count()

print("\nAUDITORIA DO MODELO DIMENSIONAL")
print("-" * 70)

print(f"Total de registros: {total:,}")
print(f"Registros sem chave de bairro: {nulos_bairro:,}")
print(f"Registros sem chave de tipo: {nulos_tipo:,}")

print("\nAMOSTRA DA FACT:")
display(
    fact_listing_dim
    .orderBy("listing_id")
    .limit(20)
)

print("\n" + "=" * 70)
print("FACT_LISTING_DIMENSIONAL CRIADA COM SUCESSO")
print("=" * 70)

In [0]:
 # ================================================================
# GOLD — TABELAS ANALÍTICAS DO MVP
# ================================================================

from pyspark.sql import functions as F

print("=" * 70)
print("CRIANDO TABELAS GOLD ANALÍTICAS")
print("=" * 70)

# ------------------------------------------------
# 1. GOLD — PREÇO POR TIPO DE ACOMODAÇÃO
# ------------------------------------------------

gold_room_type_price = (
    fact_listing
    .filter(F.col("price").isNotNull() & (F.col("price") > 0))
    .groupBy("room_type")
    .agg(
        F.count("*").alias("total_anuncios"),
        F.round(F.avg("price"), 2).alias("preco_medio"),
        F.round(F.expr("percentile_approx(price, 0.5)"), 2).alias("preco_mediano"),
        F.round(F.avg("rating"), 2).alias("avaliacao_media"),
        F.round(F.avg("availability_365"), 2).alias("disponibilidade_media")
    )
    .orderBy(F.desc("preco_medio"))
)

gold_room_type_price.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_room_type_price")


# ------------------------------------------------
# 2. GOLD — DISPONIBILIDADE
# ------------------------------------------------

gold_availability = (
    fact_listing
    .groupBy("neighbourhood_cleaned", "room_type")
    .agg(
        F.count("*").alias("total_anuncios"),
        F.round(F.avg("availability_30"), 2).alias("media_30_dias"),
        F.round(F.avg("availability_60"), 2).alias("media_60_dias"),
        F.round(F.avg("availability_90"), 2).alias("media_90_dias"),
        F.round(F.avg("availability_365"), 2).alias("media_365_dias")
    )
    .orderBy(F.desc("media_365_dias"))
)

gold_availability.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_availability")


# ------------------------------------------------
# 3. GOLD — ÍNDICE DE OPORTUNIDADE
# ------------------------------------------------

gold_opportunity = (
    fact_listing
    .filter(
        F.col("price").isNotNull() &
        (F.col("price") > 0) &
        F.col("rating").isNotNull()
    )
    .withColumn(
        "indice_oportunidade",
        F.round(
            (
                (F.col("rating") / 5) * 0.40 +
                (F.col("availability_365") / 365) * 0.30 +
                (1 / F.col("price")) * 100 * 0.30
            ),
            4
        )
    )
    .select(
        "listing_id",
        "neighbourhood_cleaned",
        "room_type",
        "price",
        "rating",
        "availability_365",
        "indice_oportunidade"
    )
    .orderBy(F.desc("indice_oportunidade"))
)

gold_opportunity.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_opportunity")


# ------------------------------------------------
# 4. VERIFICAÇÃO
# ------------------------------------------------

print("\nTABELAS GOLD CRIADAS:")

for tabela in [
    "gold_room_type_price",
    "gold_availability",
    "gold_opportunity"
]:
    df_teste = spark.table(f"workspace.default.{tabela}")
    print(
        f"✓ {tabela:<25} "
        f"{df_teste.count():>8} registros | "
        f"{len(df_teste.columns):>3} colunas"
    )

print("\n" + "=" * 70)
print("CAMADA GOLD CONCLUÍDA COM SUCESSO")
print("=" * 70)

In [0]:
 # ================================================================
# DIAGNÓSTICO — COLUNAS DA FACT_LISTING
# ================================================================

print("=" * 70)
print("COLUNAS DA FACT_LISTING")
print("=" * 70)

fact = spark.table("workspace.default.fact_listing")

for i, coluna in enumerate(fact.columns, 1):
    print(f"{i:02d}. {coluna}")

print("\n" + "=" * 70)
print("SCHEMA")
print("=" * 70)

fact.printSchema()

In [0]:
 # ================================================================
# GOLD — TABELAS ANALÍTICAS DO MVP
# VERSÃO CORRIGIDA
# ================================================================

from pyspark.sql import functions as F

print("=" * 70)
print("CRIANDO TABELAS GOLD ANALÍTICAS")
print("=" * 70)

# ================================================================
# BASE
# ================================================================

fact = spark.table("workspace.default.fact_listing")


# ================================================================
# 1. GOLD — PREÇO POR TIPO DE ACOMODAÇÃO
# ================================================================

gold_room_type_price = (
    fact
    .filter(
        F.col("price").isNotNull() &
        (F.col("price") > 0)
    )
    .groupBy("room_type")
    .agg(
        F.count("*").alias("total_anuncios"),
        F.round(F.avg("price"), 2).alias("preco_medio"),
        F.round(
            F.expr("percentile_approx(price, 0.5)"),
            2
        ).alias("preco_mediano"),
        F.round(
            F.avg("review_scores_rating"),
            2
        ).alias("avaliacao_media"),
        F.round(
            F.avg("availability_365"),
            2
        ).alias("disponibilidade_media")
    )
    .orderBy(F.desc("preco_medio"))
)

gold_room_type_price.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.gold_room_type_price"
    )

print("✓ gold_room_type_price criada")


# ================================================================
# 2. GOLD — DISPONIBILIDADE POR BAIRRO E TIPO
# ================================================================

gold_availability = (
    fact
    .groupBy(
        "neighbourhood_cleansed",
        "room_type"
    )
    .agg(
        F.count("*").alias("total_anuncios"),
        F.round(
            F.avg("availability_30"),
            2
        ).alias("media_30_dias"),
        F.round(
            F.avg("availability_60"),
            2
        ).alias("media_60_dias"),
        F.round(
            F.avg("availability_90"),
            2
        ).alias("media_90_dias"),
        F.round(
            F.avg("availability_365"),
            2
        ).alias("media_365_dias")
    )
    .orderBy(F.desc("media_365_dias"))
)

gold_availability.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.gold_availability"
    )

print("✓ gold_availability criada")


# ================================================================
# 3. GOLD — ÍNDICE DE OPORTUNIDADE
# ================================================================
#
# O índice combina:
#
#   40% → avaliação
#   30% → disponibilidade
#   30% → vantagem de preço
#
# Quanto maior o índice, mais favorável o perfil.
# ================================================================

base_opportunity = (
    fact
    .filter(
        F.col("price").isNotNull() &
        (F.col("price") > 0) &
        F.col("review_scores_rating").isNotNull()
    )
)

# Valores mínimo e máximo do preço
stats = base_opportunity.agg(
    F.min("price").alias("preco_min"),
    F.max("price").alias("preco_max")
).first()

preco_min = float(stats["preco_min"])
preco_max = float(stats["preco_max"])

print(f"Faixa de preços utilizada: R$ {preco_min:.2f} a R$ {preco_max:.2f}")

# Evita divisão por zero caso todos os preços fossem iguais
if preco_max > preco_min:

    gold_opportunity = (
        base_opportunity
        .withColumn(
            "score_avaliacao",
            F.col("review_scores_rating") / F.lit(5.0)
        )
        .withColumn(
            "score_disponibilidade",
            F.col("availability_365") / F.lit(365.0)
        )
        .withColumn(
            "score_preco",
            1 - (
                (F.col("price") - F.lit(preco_min))
                /
                F.lit(preco_max - preco_min)
            )
        )
        .withColumn(
            "indice_oportunidade",
            F.round(
                (
                    F.col("score_avaliacao") * 0.40 +
                    F.col("score_disponibilidade") * 0.30 +
                    F.col("score_preco") * 0.30
                ) * 100,
                2
            )
        )
        .select(
            "id",
            "neighbourhood_cleaned",
            "room_type",
            "price",
            "review_scores_rating",
            "availability_365",
            "indice_oportunidade"
        )
        .orderBy(
            F.desc("indice_oportunidade")
        )
    )

else:

    gold_opportunity = (
        base_opportunity
        .withColumn(
            "indice_oportunidade",
            F.round(
                (
                    (F.col("review_scores_rating") / 5.0) * 0.40 +
                    (F.col("availability_365") / 365.0) * 0.30 +
                    F.lit(0.30)
                ) * 100,
                2
            )
        )
        .select(
            "id",
            "neighbourhood_cleaned",
            "room_type",
            "price",
            "review_scores_rating",
            "availability_365",
            "indice_oportunidade"
        )
        .orderBy(
            F.desc("indice_oportunidade")
        )
    )


gold_opportunity.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.gold_opportunity"
    )

print("✓ gold_opportunity criada")


# ================================================================
# 4. VERIFICAÇÃO FINAL
# ================================================================

print("\n" + "=" * 70)
print("VERIFICAÇÃO DAS TABELAS GOLD")
print("=" * 70)

tabelas_gold = [
    "gold_neighbourhood_price",
    "gold_room_type_price",
    "gold_availability",
    "gold_opportunity"
]

for tabela in tabelas_gold:

    try:
        df_teste = spark.table(
            f"workspace.default.{tabela}"
        )

        print(
            f"✓ {tabela:<30} "
            f"{df_teste.count():>8} registros | "
            f"{len(df_teste.columns):>3} colunas"
        )

    except Exception as e:

        print(
            f"✗ {tabela:<30} NÃO ENCONTRADA"
        )

print("\n" + "=" * 70)
print("CAMADA GOLD CONCLUÍDA COM SUCESSO")
print("=" * 70)

In [0]:
 # ================================================================
# CORREÇÃO — GOLD AVAILABILITY + GOLD OPPORTUNITY
# ================================================================

from pyspark.sql import functions as F

print("=" * 70)
print("CORRIGINDO TABELAS GOLD")
print("=" * 70)

fact = spark.table("workspace.default.fact_listing")


# ================================================================
# 1. GOLD — DISPONIBILIDADE POR BAIRRO E TIPO
# ================================================================

gold_availability = (
    fact
    .groupBy(
        "neighbourhood_cleansed",
        "room_type"
    )
    .agg(
        F.count("*").alias("total_anuncios"),
        F.round(
            F.avg("availability_30"), 2
        ).alias("media_30_dias"),
        F.round(
            F.avg("availability_60"), 2
        ).alias("media_60_dias"),
        F.round(
            F.avg("availability_90"), 2
        ).alias("media_90_dias"),
        F.round(
            F.avg("availability_365"), 2
        ).alias("media_365_dias")
    )
    .orderBy(F.desc("media_365_dias"))
)

gold_availability.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.gold_availability"
    )

print("✓ gold_availability criada")


# ================================================================
# 2. GOLD — ÍNDICE DE OPORTUNIDADE
# ================================================================

base_opportunity = (
    fact
    .filter(
        F.col("price").isNotNull() &
        (F.col("price") > 0) &
        F.col("review_scores_rating").isNotNull()
    )
)

# Descobrindo os limites de preço
stats = base_opportunity.agg(
    F.min("price").alias("preco_min"),
    F.max("price").alias("preco_max")
).first()

preco_min = float(stats["preco_min"])
preco_max = float(stats["preco_max"])

print(
    f"Faixa de preços: "
    f"R$ {preco_min:.2f} a R$ {preco_max:.2f}"
)


# Normalização dos componentes do índice
gold_opportunity = (
    base_opportunity

    # Avaliação: 0 a 5 → 0 a 1
    .withColumn(
        "score_avaliacao",
        F.col("review_scores_rating") / F.lit(5.0)
    )

    # Disponibilidade: 0 a 365 → 0 a 1
    .withColumn(
        "score_disponibilidade",
        F.col("availability_365") / F.lit(365.0)
    )

    # Preço: quanto menor, maior o score
    .withColumn(
        "score_preco",
        1 - (
            (F.col("price") - F.lit(preco_min))
            /
            F.lit(preco_max - preco_min)
        )
    )

    # Índice final de 0 a 100
    .withColumn(
        "indice_oportunidade",
        F.round(
            (
                F.col("score_avaliacao") * 0.40 +
                F.col("score_disponibilidade") * 0.30 +
                F.col("score_preco") * 0.30
            ) * 100,
            2
        )
    )

    .select(
        "id",
        "neighbourhood_cleansed",
        "room_type",
        "price",
        "review_scores_rating",
        "availability_365",
        "indice_oportunidade"
    )

    .orderBy(
        F.desc("indice_oportunidade")
    )
)

gold_opportunity.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.gold_opportunity"
    )

print("✓ gold_opportunity criada")


# ================================================================
# 3. AUDITORIA FINAL DA GOLD
# ================================================================

print("\n" + "=" * 70)
print("AUDITORIA DAS TABELAS GOLD")
print("=" * 70)

tabelas_gold = [
    "gold_neighbourhood_price",
    "gold_room_type_price",
    "gold_availability",
    "gold_opportunity"
]

for tabela in tabelas_gold:

    try:
        df_teste = spark.table(
            f"workspace.default.{tabela}"
        )

        print(
            f"✓ {tabela:<30} "
            f"{df_teste.count():>8} registros | "
            f"{len(df_teste.columns):>3} colunas"
        )

    except Exception:

        print(
            f"✗ {tabela:<30} NÃO ENCONTRADA"
        )

print("\n" + "=" * 70)
print("CAMADA GOLD FINALIZADA")
print("=" * 70)

In [0]:
 # ================================================================
# GOLD — ANÁLISE DA RELAÇÃO ENTRE AVALIAÇÃO E PREÇO
# ================================================================

from pyspark.sql import functions as F

print("=" * 75)
print("CRIANDO GOLD_REVIEW_ANALYSIS")
print("=" * 75)

# Carrega a FACT principal
fact = spark.table("workspace.default.fact_listing")

# Cria faixas de avaliação
gold_review = (
    fact
    .filter(
        F.col("review_scores_rating").isNotNull() &
        F.col("price").isNotNull() &
        (F.col("review_scores_rating") > 0) &
        (F.col("price") > 0)
    )
    .withColumn(
        "faixa_avaliacao",
        F.when(F.col("review_scores_rating") < 3.5, "Abaixo de 3,5")
         .when(F.col("review_scores_rating") < 4.0, "3,5 a 3,99")
         .when(F.col("review_scores_rating") < 4.5, "4,0 a 4,49")
         .when(F.col("review_scores_rating") < 4.8, "4,5 a 4,79")
         .otherwise("4,8 a 5,0")
    )
)

# Resumo por faixa de avaliação
gold_review_analysis = (
    gold_review
    .groupBy("faixa_avaliacao")
    .agg(
        F.count("*").alias("total_anuncios"),
        F.round(F.avg("review_scores_rating"), 2).alias("avaliacao_media"),
        F.round(F.avg("price"), 2).alias("preco_medio"),
        F.round(F.expr("percentile_approx(price, 0.5)"), 2).alias("preco_mediano"),
        F.round(F.avg("number_of_reviews"), 1).alias("media_avaliacoes")
    )
    .orderBy("faixa_avaliacao")
)

# Salva a tabela GOLD
(
    gold_review_analysis.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_review_analysis")
)

print("✓ gold_review_analysis criada com sucesso")
print()
print("RESULTADO:")
display(gold_review_analysis)

print()
print("=" * 75)
print("ANÁLISE DE AVALIAÇÕES CONCLUÍDA")
print("=" * 75)

In [0]:
 # ================================================================
# ANÁLISE ESTATÍSTICA — CORRELAÇÕES DO MVP
# ================================================================

from pyspark.sql import functions as F

print("=" * 75)
print("ANÁLISE DE CORRELAÇÕES — MVP INSIDE AIRBNB SÃO PAULO")
print("=" * 75)

fact = spark.table("workspace.default.fact_listing")

# ------------------------------------------------
# 1. Correlação: capacidade x preço
# ------------------------------------------------

corr_capacidade_preco = (
    fact
    .filter(
        F.col("accommodates").isNotNull() &
        F.col("price").isNotNull() &
        (F.col("accommodates") > 0) &
        (F.col("price") > 0)
    )
    .stat.corr("accommodates", "price")
)

# ------------------------------------------------
# 2. Correlação: quartos x preço
# ------------------------------------------------

corr_quartos_preco = (
    fact
    .filter(
        F.col("bedrooms").isNotNull() &
        F.col("price").isNotNull() &
        (F.col("bedrooms") >= 0) &
        (F.col("price") > 0)
    )
    .stat.corr("bedrooms", "price")
)

# ------------------------------------------------
# 3. Correlação: camas x preço
# ------------------------------------------------

corr_camas_preco = (
    fact
    .filter(
        F.col("beds").isNotNull() &
        F.col("price").isNotNull() &
        (F.col("beds") > 0) &
        (F.col("price") > 0)
    )
    .stat.corr("beds", "price")
)

# ------------------------------------------------
# 4. Correlação: avaliação x preço
# ------------------------------------------------

corr_avaliacao_preco = (
    fact
    .filter(
        F.col("review_scores_rating").isNotNull() &
        F.col("price").isNotNull() &
        (F.col("review_scores_rating") > 0) &
        (F.col("price") > 0)
    )
    .stat.corr("review_scores_rating", "price")
)

# ------------------------------------------------
# 5. Correlação: disponibilidade x preço
# ------------------------------------------------

corr_disponibilidade_preco = (
    fact
    .filter(
        F.col("availability_365").isNotNull() &
        F.col("price").isNotNull() &
        (F.col("availability_365") >= 0) &
        (F.col("price") > 0)
    )
    .stat.corr("availability_365", "price")
)

# ------------------------------------------------
# Monta tabela de resultados
# ------------------------------------------------

correlacoes = spark.createDataFrame([
    ("Capacidade x Preço", float(corr_capacidade_preco)),
    ("Quartos x Preço", float(corr_quartos_preco)),
    ("Camas x Preço", float(corr_camas_preco)),
    ("Avaliação x Preço", float(corr_avaliacao_preco)),
    ("Disponibilidade x Preço", float(corr_disponibilidade_preco))
], ["relacao", "correlacao"])

correlacoes = correlacoes.withColumn(
    "correlacao",
    F.round("correlacao", 4)
)

display(correlacoes)

print()
print("=" * 75)
print("INTERPRETAÇÃO DA FORÇA DAS CORRELAÇÕES")
print("=" * 75)

for linha in correlacoes.collect():
    valor = linha["correlacao"]

    if abs(valor) < 0.10:
        interpretacao = "muito fraca ou praticamente inexistente"
    elif abs(valor) < 0.30:
        interpretacao = "fraca"
    elif abs(valor) < 0.50:
        interpretacao = "moderada"
    elif abs(valor) < 0.70:
        interpretacao = "forte"
    else:
        interpretacao = "muito forte"

    sentido = "positiva" if valor > 0 else "negativa"

    print(
        f"{linha['relacao']:<28} "
        f"{valor:>7.4f} → correlação {interpretacao} ({sentido})"
    )

print()
print("=" * 75)
print("ANÁLISE DE CORRELAÇÕES CONCLUÍDA")
print("=" * 75)

In [0]:
 # ================================================================
# CÉLULA 39 — CONSOLIDAÇÃO DAS RESPOSTAS DE NEGÓCIO
# ================================================================

from pyspark.sql import functions as F

print("=" * 80)
print("CONSOLIDAÇÃO DAS PERGUNTAS DE NEGÓCIO — MVP")
print("=" * 80)

# ================================================================
# Q1 — BAIRROS COM MAIOR E MENOR PREÇO MÉDIO
# ================================================================

bairro = spark.table("workspace.default.gold_neighbourhood_price")

print("\n" + "=" * 80)
print("Q1 — BAIRROS COM MAIOR E MENOR PREÇO MÉDIO")
print("=" * 80)

display(
    bairro.orderBy(
        F.col("preco_medio").desc()
    ).limit(10)
)

print("TOP 10 — MAIORES PREÇOS MÉDIOS")

display(
    bairro.orderBy(
        F.col("preco_medio").asc()
    ).limit(10)
)

print("TOP 10 — MENORES PREÇOS MÉDIOS")


# ================================================================
# Q2 — PREÇO POR TIPO DE ACOMODAÇÃO
# ================================================================

room = spark.table("workspace.default.gold_room_type_price")

print("\n" + "=" * 80)
print("Q2 — RELAÇÃO ENTRE TIPO DE ACOMODAÇÃO E PREÇO")
print("=" * 80)

display(
    room.orderBy(
        F.col("preco_medio").desc()
    )
)


# ================================================================
# Q3 — CARACTERÍSTICAS DO IMÓVEL E PREÇO
# ================================================================

print("\n" + "=" * 80)
print("Q3 — CARACTERÍSTICAS DO IMÓVEL E PREÇO")
print("=" * 80)

print("Capacidade × Preço       : 0.2111")
print("Quartos × Preço          : 0.1641")
print("Camas × Preço            : 0.1172")

print()
print("Conclusão preliminar:")
print(
    "A capacidade apresenta a maior correlação positiva com o preço, "
    "porém a relação é classificada como FRACA."
)


# ================================================================
# Q4 — AVALIAÇÃO E PREÇO
# ================================================================

print("\n" + "=" * 80)
print("Q4 — RELAÇÃO ENTRE AVALIAÇÃO E PREÇO")
print("=" * 80)

review = spark.table("workspace.default.gold_review_analysis")

display(review)

print()
print("Correlação Avaliação × Preço: 0.0018")

print(
    "Conclusão: a correlação é praticamente inexistente. "
    "Neste conjunto de dados, uma avaliação maior não está associada "
    "de forma relevante a um preço maior."
)


# ================================================================
# Q5 — DISPONIBILIDADE
# ================================================================

availability = spark.table("workspace.default.gold_availability")

print("\n" + "=" * 80)
print("Q5 — DISPONIBILIDADE DOS ANÚNCIOS")
print("=" * 80)

display(availability)


# ================================================================
# Q6 — OPORTUNIDADES
# ================================================================

opportunity = spark.table("workspace.default.gold_opportunity")

print("\n" + "=" * 80)
print("Q6 — PERFIS DE ANÚNCIOS E OPORTUNIDADES")
print("=" * 80)

display(opportunity.limit(20))

print()
print(
    "A tabela gold_opportunity reúne os indicadores utilizados "
    "para comparar preço, avaliação e disponibilidade dos anúncios."
)


# ================================================================
# RESUMO EXECUTIVO
# ================================================================

print("\n" + "=" * 80)
print("RESUMO EXECUTIVO DO MVP")
print("=" * 80)

print("""
1. PREÇO E LOCALIZAÇÃO
   O preço médio varia entre os bairros de São Paulo,
   permitindo identificar regiões com maior e menor nível de preço.

2. TIPO DE ACOMODAÇÃO
   O tipo de acomodação apresenta diferenças relevantes
   nos preços médios praticados.

3. CARACTERÍSTICAS DO IMÓVEL
   A capacidade apresenta a maior correlação com o preço,
   mas a relação ainda é fraca.

4. AVALIAÇÃO
   A correlação entre avaliação e preço é praticamente nula
   (r = 0,0018).

5. DISPONIBILIDADE
   A correlação entre disponibilidade e preço também é
   praticamente inexistente (r = 0,0213).

6. OPORTUNIDADES
   A combinação de preço, avaliação e disponibilidade permite
   criar uma visão comparativa dos perfis de anúncios.
""")

print("=" * 80)
print("CONSOLIDAÇÃO CONCLUÍDA")
print("=" * 80)

In [0]:
 # ================================================================
# AUDITORIA DE QUALIDADE DE DADOS — MVP AIRBNB SÃO PAULO
# VERSÃO CORRIGIDA
# ================================================================

from pyspark.sql import functions as F

print("=" * 80)
print("AUDITORIA DE QUALIDADE DOS DADOS")
print("=" * 80)

# ================================================================
# 1. CARREGAMENTO
# ================================================================

fact = spark.table("workspace.default.fact_listing")

total_registros = fact.count()

print(f"\nTotal de registros analisados: {total_registros:,}")
print(f"Total de atributos: {len(fact.columns)}")


# ================================================================
# 2. REGRAS DE QUALIDADE
# ================================================================

regras = {

    "id": {
        "descricao": "Identificador único do anúncio",
        "tipo": "num",
        "invalido": F.col("id") <= 0
    },

    "neighbourhood_cleaned": {
        "descricao": "Bairro do anúncio",
        "tipo": "text",
        "invalido": F.lit(False)
    },

    "room_type": {
        "descricao": "Tipo de acomodação",
        "tipo": "text",
        "invalido": ~F.col("room_type").isin(
            "Entire home/apt",
            "Private room",
            "Shared room",
            "Hotel room"
        )
    },

    "price": {
        "descricao": "Preço da diária em R$",
        "tipo": "num",
        "invalido": F.col("price") <= 0
    },

    "accommodates": {
        "descricao": "Capacidade de hóspedes",
        "tipo": "num",
        "invalido": F.col("accommodates") <= 0
    },

    "bedrooms": {
        "descricao": "Quantidade de quartos",
        "tipo": "num",
        "invalido": F.col("bedrooms") < 0
    },

    "beds": {
        "descricao": "Quantidade de camas",
        "tipo": "num",
        "invalido": F.col("beds") < 0
    },

    "availability_30": {
        "descricao": "Dias disponíveis nos próximos 30 dias",
        "tipo": "num",
        "invalido": (
            (F.col("availability_30") < 0) |
            (F.col("availability_30") > 30)
        )
    },

    "availability_60": {
        "descricao": "Dias disponíveis nos próximos 60 dias",
        "tipo": "num",
        "invalido": (
            (F.col("availability_60") < 0) |
            (F.col("availability_60") > 60)
        )
    },

    "availability_90": {
        "descricao": "Dias disponíveis nos próximos 90 dias",
        "tipo": "num",
        "invalido": (
            (F.col("availability_90") < 0) |
            (F.col("availability_90") > 90)
        )
    },

    "availability_365": {
        "descricao": "Dias disponíveis nos próximos 365 dias",
        "tipo": "num",
        "invalido": (
            (F.col("availability_365") < 0) |
            (F.col("availability_365") > 365)
        )
    },

    "number_of_reviews": {
        "descricao": "Quantidade de avaliações",
        "tipo": "num",
        "invalido": F.col("number_of_reviews") < 0
    },

    "review_scores_rating": {
        "descricao": "Nota média das avaliações",
        "tipo": "num",
        "invalido": (
            (F.col("review_scores_rating") < 0) |
            (F.col("review_scores_rating") > 5)
        )
    },

    "host_is_superhost": {
        "descricao": "Indicador de Superhost",
        "tipo": "text",
        "invalido": ~F.col("host_is_superhost").isin("t", "f")
    },

    "instant_bookable": {
        "descricao": "Permite reserva instantânea",
        "tipo": "text",
        "invalido": ~F.col("instant_bookable").isin("t", "f")
    },

    "last_scraped": {
        "descricao": "Data da última coleta",
        "tipo": "date",
        "invalido": F.lit(False)
    }
}


# ================================================================
# 3. AUDITORIA
# ================================================================

colunas_existentes = set(fact.columns)

resultados = []

for coluna, regra in regras.items():

    if coluna not in colunas_existentes:
        continue

    # ------------------------------------------------------------
    # NULOS
    # ------------------------------------------------------------

    nulos = fact.filter(
        F.col(coluna).isNull()
    ).count()

    # ------------------------------------------------------------
    # VALORES VAZIOS
    # ------------------------------------------------------------

    if regra["tipo"] == "text":

        vazios = fact.filter(
            F.col(coluna).isNotNull() &
            (F.trim(F.col(coluna)) == "")
        ).count()

    else:

        vazios = 0

    # ------------------------------------------------------------
    # VALORES INVÁLIDOS
    # ------------------------------------------------------------

    invalidos = fact.filter(
        F.col(coluna).isNotNull() &
        regra["invalido"]
    ).count()

    # ------------------------------------------------------------
    # TOTAL DE PROBLEMAS
    # ------------------------------------------------------------

    problemas = nulos + vazios + invalidos

    percentual_nulos = (
        100 * nulos / total_registros
    )

    percentual_invalidos = (
        100 * invalidos / total_registros
    )

    percentual_qualidade = max(
        0,
        100 * (1 - problemas / total_registros)
    )

    # ------------------------------------------------------------
    # MÍNIMO E MÁXIMO
    # Agora armazenados como STRING para evitar erro Arrow
    # ------------------------------------------------------------

    valor_minimo = ""
    valor_maximo = ""

    if regra["tipo"] == "num":

        valores = fact.select(
            F.min(F.col(coluna)).alias("minimo"),
            F.max(F.col(coluna)).alias("maximo")
        ).collect()[0]

        if valores["minimo"] is not None:
            valor_minimo = str(valores["minimo"])

        if valores["maximo"] is not None:
            valor_maximo = str(valores["maximo"])

    # ------------------------------------------------------------
    # CLASSIFICAÇÃO
    # ------------------------------------------------------------

    if percentual_qualidade >= 99:
        classificacao = "EXCELENTE"

    elif percentual_qualidade >= 95:
        classificacao = "BOA"

    elif percentual_qualidade >= 90:
        classificacao = "ACEITÁVEL"

    else:
        classificacao = "ATENÇÃO"

    resultados.append((
        coluna,
        regra["descricao"],
        total_registros,
        nulos,
        round(percentual_nulos, 2),
        vazios,
        invalidos,
        round(percentual_invalidos, 2),
        valor_minimo,
        valor_maximo,
        round(percentual_qualidade, 2),
        classificacao
    ))


# ================================================================
# 4. DATAFRAME DE QUALIDADE
# ================================================================

schema = """
atributo string,
descricao string,
total_registros long,
nulos long,
percentual_nulos double,
vazios long,
invalidos long,
percentual_invalidos double,
valor_minimo string,
valor_maximo string,
percentual_qualidade double,
classificacao string
"""

df_quality = spark.createDataFrame(
    resultados,
    schema=schema
)


# ================================================================
# 5. ORDENAÇÃO
# ================================================================

df_quality = df_quality.orderBy(
    F.col("percentual_qualidade").asc()
)


# ================================================================
# 6. RESULTADO
# ================================================================

print("\n")
print("=" * 80)
print("RESULTADO DA AUDITORIA POR ATRIBUTO")
print("=" * 80)

display(df_quality)


# ================================================================
# 7. AUDITORIA DA CHAVE PRIMÁRIA
# ================================================================

print("\n")
print("=" * 80)
print("AUDITORIA DA CHAVE PRIMÁRIA")
print("=" * 80)

total_ids = fact.select("id").count()

ids_distintos = fact.select("id").distinct().count()

ids_duplicados = total_ids - ids_distintos

print(f"Total de registros:      {total_ids:,}")
print(f"IDs distintos:           {ids_distintos:,}")
print(f"Registros duplicados:    {ids_duplicados:,}")

if ids_duplicados == 0:
    print("✓ CHAVE PRIMÁRIA ÍNTEGRA — nenhum ID duplicado.")
else:
    print("⚠ ATENÇÃO — existem IDs duplicados.")


# ================================================================
# 8. RESUMO EXECUTIVO
# ================================================================

atributos_auditados = df_quality.count()

atributos_excelentes = df_quality.filter(
    F.col("classificacao") == "EXCELENTE"
).count()

atributos_atencao = df_quality.filter(
    F.col("classificacao") == "ATENÇÃO"
).count()

qualidade_media = df_quality.select(
    F.avg("percentual_qualidade")
).collect()[0][0]

print("\n")
print("=" * 80)
print("RESUMO EXECUTIVO DA QUALIDADE")
print("=" * 80)

print(f"Registros analisados:          {total_registros:,}")
print(f"Atributos auditados:           {atributos_auditados}")
print(f"Qualidade média:               {qualidade_media:.2f}%")
print(f"Atributos excelentes:          {atributos_excelentes}")
print(f"Atributos que exigem atenção:  {atributos_atencao}")

if ids_duplicados == 0:
    print("Integridade da chave:          ✓ APROVADA")
else:
    print("Integridade da chave:          ⚠ REQUER TRATAMENTO")

print("=" * 80)
print("AUDITORIA CONCLUÍDA COM SUCESSO")
print("=" * 80)

In [0]:
 # ============================================================
# ANALISE FINAL DO MVP
# Respostas às perguntas de negócio
# ============================================================

from pyspark.sql import functions as F

print("=" * 80)
print("ANÁLISE FINAL — MVP AIRBNB SÃO PAULO")
print("=" * 80)

# ------------------------------------------------------------
# 1. PREÇO POR BAIRRO
# ------------------------------------------------------------

print("\n1. PREÇO MÉDIO POR BAIRRO")
print("-" * 80)

bairro = spark.table("workspace.default.gold_neighbourhood_price")

display(
    bairro
    .orderBy(F.desc("preco_medio"))
    .limit(10)
)

print("TOP 10 BAIRROS COM MAIOR PREÇO MÉDIO")


# ------------------------------------------------------------
# 2. PREÇO POR TIPO DE ACOMODAÇÃO
# ------------------------------------------------------------

print("\n2. PREÇO MÉDIO POR TIPO DE ACOMODAÇÃO")
print("-" * 80)

tipo = spark.table("workspace.default.gold_room_type_price")

display(
    tipo
    .orderBy(F.desc("preco_medio"))
)

print("COMPARAÇÃO ENTRE OS TIPOS DE ACOMODAÇÃO")


# ------------------------------------------------------------
# 3. CARACTERÍSTICAS DO IMÓVEL X PREÇO
# ------------------------------------------------------------

print("\n3. CARACTERÍSTICAS DO IMÓVEL X PREÇO")
print("-" * 80)

oportunidade = spark.table("workspace.default.gold_opportunity")

display(
    oportunidade.limit(20)
)

print("""
Esta análise permite observar conjuntamente:
- capacidade do imóvel;
- preço;
- avaliação;
- disponibilidade.
""")


# ------------------------------------------------------------
# 4. AVALIAÇÃO X PREÇO
# ------------------------------------------------------------

print("\n4. AVALIAÇÃO X PREÇO")
print("-" * 80)

avaliacao = spark.table("workspace.default.gold_review_analysis")

display(avaliacao)

print("""
A tabela apresenta a relação entre faixas de avaliação,
quantidade de anúncios e preços médios.
""")


# ------------------------------------------------------------
# 5. DISPONIBILIDADE
# ------------------------------------------------------------

print("\n5. DISPONIBILIDADE DOS ANÚNCIOS")
print("-" * 80)

disponibilidade = spark.table("workspace.default.gold_availability")

display(
    disponibilidade.limit(30)
)

print("""
A disponibilidade é analisada considerando bairro e
tipo de acomodação.
""")


# ------------------------------------------------------------
# 6. OPORTUNIDADE
# ------------------------------------------------------------

print("\n6. PERFIS COM MELHOR COMBINAÇÃO DE PREÇO, AVALIAÇÃO E DISPONIBILIDADE")
print("-" * 80)

display(
    oportunidade
    .orderBy(
        F.desc("review_scores_rating"),
        F.asc("price")
    )
    .limit(20)
)

print("=" * 80)
print("ANÁLISE FINAL CONCLUÍDA")
print("=" * 80)

In [0]:
 # ============================================================
# PAINEL EXECUTIVO — MVP AIRBNB SÃO PAULO
# ============================================================

from pyspark.sql import functions as F

print("=" * 85)
print("           PAINEL EXECUTIVO — AIRBNB SÃO PAULO")
print("=" * 85)

# ------------------------------------------------------------
# INDICADOR GERAL
# ------------------------------------------------------------

fact = spark.table("workspace.default.fact_listing")

total = fact.count()

print(f"\nTOTAL DE ANÚNCIOS ANALISADOS: {total:,}".replace(",", "."))


# ------------------------------------------------------------
# 1. BAIRROS MAIS CAROS
# ------------------------------------------------------------

print("\n" + "=" * 85)
print("1. BAIRROS COM MAIOR PREÇO MÉDIO")
print("=" * 85)

bairro = spark.table("workspace.default.gold_neighbourhood_price")

display(
    bairro
    .orderBy(F.desc("preco_medio"))
    .limit(10)
)


# ------------------------------------------------------------
# 2. TIPOS DE ACOMODAÇÃO
# ------------------------------------------------------------

print("\n" + "=" * 85)
print("2. PREÇO MÉDIO POR TIPO DE ACOMODAÇÃO")
print("=" * 85)

room = spark.table("workspace.default.gold_room_type_price")

display(
    room.orderBy(F.desc("preco_medio"))
)


# ------------------------------------------------------------
# 3. DISPONIBILIDADE
# ------------------------------------------------------------

print("\n" + "=" * 85)
print("3. DISPONIBILIDADE POR BAIRRO E TIPO")
print("=" * 85)

availability = spark.table("workspace.default.gold_availability")

display(
    availability.limit(30)
)


# ------------------------------------------------------------
# 4. AVALIAÇÕES
# ------------------------------------------------------------

print("\n" + "=" * 85)
print("4. AVALIAÇÃO X PREÇO")
print("=" * 85)

reviews = spark.table("workspace.default.gold_review_analysis")

display(reviews)


# ------------------------------------------------------------
# 5. OPORTUNIDADES
# ------------------------------------------------------------

print("\n" + "=" * 85)
print("5. MELHORES OPORTUNIDADES")
print("=" * 85)

opportunity = spark.table("workspace.default.gold_opportunity")

display(
    opportunity
    .orderBy(
        F.desc("review_scores_rating"),
        F.asc("price")
    )
    .limit(20)
)


# ------------------------------------------------------------
# ENCERRAMENTO
# ------------------------------------------------------------

print("\n" + "=" * 85)
print("PAINEL EXECUTIVO GERADO COM SUCESSO")
print("=" * 85)